<a href="https://colab.research.google.com/github/ualibraries/ual-libapps/blob/master/PythonRevisedFeb11_API.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [18]:
import google.generativeai as genai
import pandas as pd
import time
import json
from google.api_core import exceptions
from google.colab import userdata
import re # Added import for re module
import os # Added import for os module




# --- 1. CONFIGURATION ---
API_KEY = userdata.get('GEMINI_API_KEY')
INPUT_FILE = "test3_Transcripts.csv"
OUTPUT_FILE = "coded_results_pilot.csv"
MAX_ROWS = 100
SAVE_INTERVAL = 10 # Added definition for SAVE_INTERVAL


genai.configure(api_key=API_KEY)
model = genai.GenerativeModel('gemini-flash-latest')


# --- 2. CODEBOOK & SYSTEM PROMPT ---
# We extract keys here so the Python script can "check" the AI's homework
CODEBOOK_DICT = {
  "Abandoned Chat": "Empty fields, greetings only, or just 'library help' with no intent.",
  "Request Purchase": "Requests for library to acquire books, journals, databases or other materials. request may be for personal use, course support, or research purposes. ",
  "Hours": "Opening/closing times of the library or specific departments.",
  "Navigation & Wayfinding": "Locating areas, physical items, collections, restrooms or services.",
  "Lost & Found": "Inquiries about lost personal items.",
  "Noise Issues": "Concerns about noise levels or quiet areas.",
  "Physical Accessibility": "Inquiries about ADA access, ADA software or elevators.",
  "Safety & Security": "Personal safety or security, emergency procedures, or the presence of security personnel.",
  "Study Rooms & Reservations": "Booking or availability of study rooms including availability, policies, and equipment.",
  "Campus: Services": "non-library services offered on campus (e.g., tutoring, writing support,  IT help, financial aid, counseling, dining).",
  "Campus Wayfinding": "Directions to locations elsewhere on campus.",
  "Course Reserves": "Borrowing textbooks/readings for classes.",
  "Faculty Instruction Support": "faculty or instructors seeking help with teaching-related resources, tools, or services..",
  "Known Item: AV": "Specific DVDs, streaming, audiobooks, or music.",
  "Known Item: Books": "Specific books by title, call number, or ISBN.",
  "Known Item: Articles": "Specific journal titles or individual articles.",
  "Known Item: Thesis": "Specific academic theses or dissertations.",
  "Known Item: Other": "Kits, maps, tools, games, maps or anatomy boxes.",
  "Known Item: Author": "Works by a particular author.",
  "Interlibrary Loan": "Borrowing from other library systems.",
  "Library Services": "General info on workshops or reference help (e.g., reference help, workshops).",
  "Fines & Fees": "Fines or payment procedures.",
  "Holds Request": "Placing or managing requests for items.",
  "Lost Items": "Items borrowed and lost by the patron.",
  "Patron Account": "Logins, library cards, or account status.",
  "Renewals": "Extending borrowing periods.",
  "Policies & Procedures": "library rules, borrowing limits, conduct policies, or service guidelines.",
  "Citations & Citing Sources": "citation styles (APA, MLA, etc.), tools, or avoiding plagiarism.",
  "Database Search Skills": "Using search techniques, filters or Boolean operators.",
  "Develop Research Topic": "Brainstorming, narrowing or refining a research question.",
  "Evaluation Information": "Assessing credibility or quality of sources.",
  "Finding Relevant Resources": "Identifying topic-specific materials.",
  "Managing & Organizing Information": "Organizing research notes and references.",
  "Research Strategy": "Broad help on how to begin or improve research.",
  "Borrow Tech": "Laptops, chargers, tablets, cameras or other hardware loans.",
  "Connectivity & Remote Access Issues": "VPN, Wi-Fi, or remote access issues.",
  "Software": "Software availability, installation or use (e.g. Use of SPSS, Adobe, etc).",
  "Tech Support": "Help with frozen computers or equipment login.",
  "Website": "Issues or questions navigating the library site or scheduling appointments.",
  "Printing & Scanning": "Printing, scanning, or reporting equipment issues. 3D print",
  "Other": "Inquiries not fitting in any above category."
}


VALID_CODES = set(CODEBOOK_DICT.keys())

SYSTEM_PROMPT = f"""
Role: Senior Library Science Researcher.
Task: Multi-label coding of transcripts using ONLY the provided JSON.

 # PILLAR 1: THE CLOSED-LIST RULE (Discipline)
- NO INVENTED CODES: Use ONLY keys from the JSON. You are legally barred from inventing codes.
- If a concept is "good" but not in the JSON, you MUST map it to the closest existing category or use 'Other'.
 # PILLAR 2: PRIMARY INTENT (The "Kitchen Sink" Fix)
- Only code for the patron's actual NEED.
- Ignore "Atmospheric Keywords." If they say "I'm in the library (Navigation) looking at the website (Website) to find a book (Books)," the ONLY code is 'Books'.
 # PILLAR 3: REASONING AS AUDIT (The "Gem" Intuition)
- Use the reasoning section to explain why you chose a specific category over a similar one. This captures the "insight" you want without breaking the codebook.

# CODING PROTOCOL:
1.  Identify ALL relevant codes from the JSON
2.  KEYWORD CONTEXTUALIZATION: Map keywords to Intent and Definition. Do not infer meaning
3.  MULTI-LABELING: Assign ALL relevant codes if a transcript touches multiple topics. Separate with commas
4. ABANDONED CHAT LOGIC (STRICT):
- Code as 'Abandoned Chat' if field is blank, [empty], or exclusively greetings ("hi", "hello")
- THRESHOLD RULE: If only content is "library help" or "library chat service" with no other nouns/verbs, code as 'Abandoned Chat'
- DO NOT code as Abandoned if second layer of intent exists (e.g., "login question")
5. EXCLUSION: Ignore system tags like <url> or <person>
6. CONSTRAINT: No prose or summaries. Process every ID individually




# OUTPUT FORMAT:
Code1, Code2 | Brief reasoning (1 sentence).




# CODEBOOK JSON:
{json.dumps(CODEBOOK_DICT, indent=2)}
"""




# --- 3. UTILITY FUNCTIONS ---
def clean_raw_text(text):
    if pd.isna(text): return ""
    # Strip clock times and specific date/time tags
    text = re.sub(r'\d{2}:\d{2}:\d{2}', '', text)
    text = re.sub(r'<DATE_TIME>', '', text)
    # Optional: Strip the long alphanumeric person IDs to save tokens
    text = re.sub(r'[a-f0-9]{32,}', 'STAFF', text)
    return text.strip()

def code_transcript(transcript):
    cleaned_input = clean_raw_text(transcript)
    if len(cleaned_input) < 10:
        return "Abandoned Chat | Insufficient data"

    for attempt in range(2): # 2 attempts per row to keep things moving
        try:
            response = model.generate_content(
                f"{SYSTEM_PROMPT}\n\nTranscript: {cleaned_input}",
                generation_config={"temperature": 0.0}
            )
            return response.text.replace("**", "").replace("\n", " ").strip()
        except Exception as e:
            time.sleep(2)
    return f"ERROR | {str(e)[:50]}"

# --- 4. MAIN EXECUTION LOOP ---
def main():
    # Check if we are resuming from an existing output file
    if os.path.exists(OUTPUT_FILE):
        print(f"Resuming from {OUTPUT_FILE}...")
        df = pd.read_csv(OUTPUT_FILE)
    else:
        print(f"Starting fresh from {INPUT_FILE}...")
        df = pd.read_csv(INPUT_FILE)

    if 'Applied_Code_Reasoning' not in df.columns:
        df['Applied_Code_Reasoning'] = ""

    total_rows = len(df)
    processed_this_session = 0

    print(f"Beginning processing of {total_rows} rows...")

    try:
        for i, row in df.iterrows():
            # SKIP if already coded (The Resume Logic)
            if pd.notnull(df.at[i, 'Applied_Code_Reasoning']) and df.at[i, 'Applied_Code_Reasoning'] != "":
                continue

            # API Call
            result = code_transcript(row['OriginalTranscript'])
            df.at[i, 'Applied_Code_Reasoning'] = result

            processed_this_session += 1

            # Print progress to console
            if processed_this_session % 10 == 0:
                print(f"Progress: {i+1}/{total_rows} rows completed...")

            # Periodic Save
            if processed_this_session % SAVE_INTERVAL == 0:
                df.to_csv(OUTPUT_FILE, index=False)
                print(f"--- Intermediate save at row {i+1} ---")

            # Rate limiting safety (0.5s)
            time.sleep(0.5)

    except KeyboardInterrupt:
        print("\nManual stop detected. Saving progress...")
    finally:
        df.to_csv(OUTPUT_FILE, index=False)
        print(f"\nBatch complete. Total session count: {processed_this_session}")

if __name__ == "__main__":
    main()


Starting fresh from test3_Transcripts.csv...
Beginning processing of 19 rows...
Progress: 10/19 rows completed...
--- Intermediate save at row 10 ---

Batch complete. Total session count: 19
